# Pipeline de données RAW pour la suppression de reflets

Reproduction **simplifiée** de la génération de données de *Removing Reflections
from RAW Photos* (Kee, Pikielny, Blackburn-Matzen, Levoy — CVPR 2025), en
utilisant uniquement le dataset **MIT-Adobe FiveK**.

**Idée clé du papier** : la lumière s'additionne linéairement sur le capteur.
Un reflet dans une vitre est donc un mélange `m = t + r` (transmission +
réflexion) — mais cette addition n'est valable que sur des images
*scene-referred* (linéaires par rapport à la luminance de la scène), **pas**
sur des JPEG tone-mappés. Le papier simule donc des reflets en additionnant
des photos RAW converties en XYZ linéaire, *avant* balance des blancs, pour
que les couleurs des illuminants se mélangent correctement.

| Étape du papier | Ici |
|---|---|
| Sources : MIT5K + RAISE + panoramas Laval | MIT-Adobe FiveK uniquement |
| ACR étapes 1–4 : linéarisation, dématriçage, niveau de noir, → XYZ | `rawpy` (LibRaw), RGB capteur → XYZ via la `ColorMatrix` du DNG |
| Dé-exposition `e = s·g/n²` (Sec. 3.1) | EXIF du DNG via `exifread` |
| Simulation géométrique : Fresnel par rayon, perspective, double réflexion, défocus calibré (Sec. 3.2 / B) | caméra + vitre échantillonnées : Fresnel exact par pixel, homographie de rotation, fantôme et défocus dérivés de l'optique |
| Photo contextuelle `c` = moitié disjointe de l'image reflet (Sec. 3.3) | idem |
| `m = t + r`, ré-exposition `e' = τ/E[m']`, WB partagée dans XYZ (Func. 1) | idem, WB = illuminant *as-shot* de la transmission (CAT Bradford → D50) |
| Recherche de mélanges « utiles » parmi 10⁸ candidats + priors sémantiques (Sec. D) | scoring de paires réalistes (catégories FiveK + illuminants) + heuristiques de filtrage |
| Sortie : (m, t, r, c) en sRGB linéaire, 256p et 2048p | simulation en linéaire, puis (m, t, r, c) **finis en JPEG sRGB** 256p via notre ISP |

**Format de sortie** : la physique (`m = t + r`) n'a besoin d'être vraie
qu'*au moment de la synthèse*, en linéaire. Le dataset final est ensuite
« fini » par une pipeline ISP (tone mapping + gamma) et sauvé en **JPEG
sRGB** — des images qui ressemblent à de vraies photos JPEG, mais dont le
mélange sous-jacent était photométriquement correct. C'est l'inverse des
datasets 8 bits critiqués par le papier, qui *mélangent* directement des
images tone-mappées.


In [ ]:
# Dépendances (rawpy = lecture RAW/LibRaw, exifread = métadonnées EXIF des DNG)
%pip install --quiet rawpy exifread requests tqdm opencv-python matplotlib numpy


In [ ]:
import json
import random
from pathlib import Path

import numpy as np

DATA_DIR = Path("fivek_data")            # relatif au notebook
DNG_DIR = DATA_DIR / "dng"
OUT_DIR = DATA_DIR / "simulated"
for d in (DNG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

N_OUTDOOR = 8          # nb de DNG "outdoor" à télécharger (~5-30 Mo chacun)
N_INDOOR = 8           # nb de DNG "indoor"
N_EXAMPLES = 32        # nb d'exemples simulés (m, t, r, c) à générer
JPEG_QUALITY = 95      # qualité des JPEG du dataset
SAVE_LINEAR = False    # True pour sauver aussi les .npz sRGB linéaire
PATCH = 256            # résolution des exemples (le papier travaille à 256p)
TAU = 0.18             # exposition cible du pixel moyen (gris moyen, Func. S1)
MAX_SIDE = 1536        # taille max des RAW gardés en mémoire
NORMALIZE_POWER = True # voir la section simulation : ratio de puissance t/r
SEED = 0

rng = np.random.default_rng(SEED)
rnd = random.Random(SEED)


## 1. Téléchargement d'un sous-ensemble de MIT-Adobe FiveK

On s'appuie sur les métadonnées JSON du repo
[yuukicammy/mit-adobe-fivek-dataset](https://github.com/yuukicammy/mit-adobe-fivek-dataset)
(hébergées sur Hugging Face), qui référencent les 5 000 DNG officiels de
`data.csail.mit.edu` avec leurs catégories (`location`, `time`, `light`,
`subject`) — on s'en servira pour choisir des paires réalistes.

Le papier apparie des scènes **intérieures et extérieures** (une vitre sépare
presque toujours un intérieur d'un extérieur, et leurs illuminants diffèrent) :
on télécharge donc les deux catégories. Augmente `N_OUTDOOR` / `N_INDOOR` pour
un vrai dataset — ici on reste petit pour que le notebook tourne vite.


In [ ]:
import requests
from tqdm.auto import tqdm

META_URL = "https://huggingface.co/datasets/yuukicammy/MIT-Adobe-FiveK/raw/main/training.json"

meta_path = DATA_DIR / "training.json"
if not meta_path.exists():
    resp = requests.get(META_URL, timeout=120)
    resp.raise_for_status()
    meta_path.write_bytes(resp.content)
metadata = json.loads(meta_path.read_text())
print(f"{len(metadata)} images référencées")

by_loc = {}
for name, item in metadata.items():
    loc = item.get("categories", {}).get("location", "unknown")
    by_loc.setdefault(loc, []).append((name, item))
print({k: len(v) for k, v in sorted(by_loc.items())})

selection = []
for loc, n in [("outdoor", N_OUTDOOR), ("indoor", N_INDOOR)]:
    items = list(by_loc.get(loc, []))
    rnd.shuffle(items)
    selection += [(name, item, loc) for name, item in items[:n]]


def download_dng(name, item):
    url = item["urls"]["dng"].replace("http://", "https://")
    path = DNG_DIR / f"{name}.dng"
    if path.exists() and path.stat().st_size > 0:
        return path
    with requests.get(url, stream=True, timeout=300) as resp:
        resp.raise_for_status()
        tmp = path.with_suffix(".part")
        with open(tmp, "wb") as f:
            for chunk in resp.iter_content(1 << 20):
                f.write(chunk)
        tmp.rename(path)
    return path


dng_files = []  # liste de (chemin, location, catégories)
for name, item, loc in tqdm(selection, desc="DNG"):
    try:
        dng_files.append((download_dng(name, item), loc,
                          item.get("categories", {})))
    except Exception as exc:
        print(f"échec {name}: {exc}")

total_mb = sum(p.stat().st_size for p, _, _ in dng_files) / 1e6
print(f"{len(dng_files)} DNG prêts ({total_mb:.0f} Mo)")


## 2. RAW → XYZ linéaire (ACR étapes 1–4)

Le papier montre que l'étape idéale pour mélanger deux photos est la sortie de
l'étape 4 du pipeline Adobe Camera Raw : **XYZ linéaire, avant balance des
blancs**. À ce stade :

- les pixels sont proportionnels à la luminance de la scène (on peut additionner) ;
- l'espace couleur est indépendant du capteur (on peut mélanger deux appareils) ;
- la **couleur de l'illuminant est préservée** — crucial, car la scène
  transmise et la scène reflétée sont en général éclairées différemment, et
  leurs dominantes se mélangent *avant* la balance des blancs.

Avec `rawpy` : `gamma=(1,1)` (pas de compression), `no_auto_bright`,
`user_wb=(1,1,1,1)` (désactive la WB), sortie en **RGB capteur** puis
conversion explicite vers XYZ en inversant la `ColorMatrix` du DNG
(c'est littéralement l'étape 4 d'ACR ; le papier suit la spec DNG exacte,
Func. S4-S9). On lit aussi :

- l'exposition EXIF `e = s·g/n²` (temps de pose × ISO / ouverture²) pour
  « dé-exposer » les images (Sec. 3.1) ;
- l'**illuminant as-shot** du DNG (balance des blancs mesurée par l'appareil),
  converti en XYZ — il servira pour la balance des blancs (Func. S2) et pour
  le scoring des paires.

**Pixels saturés** : un pixel écrêté sur le capteur a une vraie couleur
inconnue ; converti naïvement en XYZ sans WB il devient magenta. On lui donne
la chromaticité de l'illuminant — il redeviendra blanc après balance des
blancs, comme dans une vraie ISP (le papier a une règle dédiée pour que les
pixels saturés *restent* saturés, Func. S1).


In [ ]:
import exifread
import rawpy


def read_exposure(path):
    """Exposition e = s * g / n**2 (Sec. 3.1 du papier)."""
    with open(path, "rb") as f:
        tags = exifread.process_file(f, details=False)

    def tag(name, default):
        t = tags.get(name)
        if t is None or not t.values:
            return default
        v = t.values[0]
        try:
            return float(v.num) / float(v.den)
        except AttributeError:
            return float(v)

    s = tag("EXIF ExposureTime", 1 / 60)          # temps de pose (s)
    n = tag("EXIF FNumber", 4.0)                  # ouverture
    g = tag("EXIF ISOSpeedRatings", 100.0)        # gain (ISO)
    return s * g / max(n, 0.7) ** 2


def read_raw_to_xyz(path, half_size=True):
    """ACR étapes 1-4 : linéarisation, dématriçage, niveau de noir, -> XYZ.

    user_wb=(1,1,1,1) désactive la balance des blancs : on obtient un XYZ
    "as-shot" où la couleur de l'illuminant est préservée.
    Retourne (xyz, illuminant_xyz) ; illuminant = None si le DNG n'a pas de
    balance des blancs as-shot exploitable.
    """
    with rawpy.imread(str(path)) as raw:
        cam = raw.postprocess(
            gamma=(1, 1),
            no_auto_bright=True,
            output_bps=16,
            use_camera_wb=False,
            use_auto_wb=False,
            user_wb=[1.0, 1.0, 1.0, 1.0],
            output_color=rawpy.ColorSpace.raw,  # RGB capteur, dématriçé, sans WB
            half_size=half_size,  # binning 2x2 : plus rapide, suffisant pour 256p
        )
        # ColorMatrix du DNG : XYZ -> RGB capteur ; on l'inverse (étape 4 ACR)
        cam_to_xyz = np.linalg.inv(raw.rgb_xyz_matrix[:3, :3]).astype(np.float32)
        wb = np.array(raw.camera_whitebalance[:3], np.float32)

    # illuminant as-shot : l'appareil neutralise l'illuminant en multipliant
    # par wb ; la réponse du capteur à l'illuminant est donc proportionnelle
    # à 1/wb, qu'on remonte en XYZ
    illum = None
    if np.all(wb > 0):
        illum = cam_to_xyz @ (wb[1] / wb)
        illum = (illum / max(illum[1], 1e-8)).astype(np.float32)

    cam = cam.astype(np.float32) / 65535.0
    saturated = cam.max(axis=-1) >= 0.98        # pixels écrêtés sur le capteur
    xyz = np.einsum("ij,hwj->hwi", cam_to_xyz, cam)

    if saturated.any() and not saturated.all():
        # vraie couleur inconnue -> chromaticité de l'illuminant,
        # pour qu'ils redeviennent blancs après balance des blancs
        if illum is not None:
            ref = illum
        else:
            ref = xyz[~saturated].reshape(-1, 3).mean(0)
            ref = ref / max(ref[1], 1e-8)
        xyz[saturated] = ref * xyz[saturated][:, 1:2]
    return np.clip(xyz, 0, None), illum


In [ ]:
import matplotlib.pyplot as plt

# --- Outils couleur -------------------------------------------------------
D50 = np.array([0.9642, 1.0, 0.8249], np.float32)

BRADFORD = np.array([[0.8951, 0.2664, -0.1614],
                     [-0.7502, 1.7135, 0.0367],
                     [0.0389, -0.0685, 1.0296]], np.float32)


def cat_matrix(src_white, dst_white=D50):
    """Adaptation chromatique de Bradford : XYZ(illuminant src) -> XYZ(dst)."""
    s = BRADFORD @ src_white
    d = BRADFORD @ dst_white
    return (np.linalg.inv(BRADFORD) @ np.diag(d / s) @ BRADFORD).astype(np.float32)


# Matrices standard (ICC / IEC 61966-2-1)
XYZ_D50_TO_SRGB = np.array([[3.1338561, -1.6168667, -0.4906146],
                            [-0.9787684, 1.9161415, 0.0334540],
                            [0.0719453, -0.2289914, 1.4052427]], np.float32)
XYZ_D65_TO_SRGB = np.array([[3.2404542, -1.5371385, -0.4985314],
                            [-0.9692660, 1.8760108, 0.0415560],
                            [0.0556434, -0.2040259, 1.0572252]], np.float32)


def apply_matrix(img, M):
    return np.einsum("ij,...j->...i", M, img)


def gray_world_white(xyz):
    """Estimation gray-world de l'illuminant (fallback si pas d'as-shot)."""
    w = xyz.reshape(-1, 3).mean(0)
    return (w / max(w[1], 1e-8)).astype(np.float32)


def luminance(lin_srgb):
    return lin_srgb @ np.array([0.2126, 0.7152, 0.0722], np.float32)


def srgb_encode(x):
    """Gamma sRGB (linéaire [0,1] -> encodé [0,1])."""
    x = np.clip(x, 0.0, 1.0)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)


# --- Affichage ------------------------------------------------------------
def auto_expose(lin, target=0.18):
    """Expose la moyenne géométrique de la luminance sur le gris moyen."""
    key = float(np.exp(np.log(np.clip(luminance(lin), 1e-6, None)).mean()))
    return lin * (target / max(key, 1e-8))


def to_display(lin_srgb, ev=0.0):
    return srgb_encode(np.clip(lin_srgb * 2.0 ** ev, 0, 1))


def xyz_to_display(xyz, illum=None, white_balance=True):
    if white_balance:
        w = illum if illum is not None else gray_world_white(xyz)
        lin = apply_matrix(xyz, XYZ_D50_TO_SRGB @ cat_matrix(w))
    else:
        lin = apply_matrix(xyz, XYZ_D65_TO_SRGB)
    return np.clip(auto_expose(np.clip(lin, 0, None)), 0, 1)


def show_row(images, titles, ev=0.0, size=3.2):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, img, title in zip(np.atleast_1d(axes), images, titles):
        ax.imshow(to_display(img, ev))
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# Aperçu : le même RAW sans / avec balance des blancs (illuminant as-shot).
# Sans WB, la dominante de l'illuminant est visible (tungstène orangé en
# intérieur, bleu du ciel en extérieur) : c'est cette information que le
# papier tient à préserver au moment du mélange.
p_out = next(p for p, loc, _ in dng_files if loc == "outdoor")
p_in = next(p for p, loc, _ in dng_files if loc == "indoor")
xyz_out, il_out = read_raw_to_xyz(p_out)
xyz_in, il_in = read_raw_to_xyz(p_in)
show_row([xyz_to_display(xyz_out, white_balance=False), xyz_to_display(xyz_out, il_out),
          xyz_to_display(xyz_in, white_balance=False), xyz_to_display(xyz_in, il_in)],
         ["outdoor — XYZ sans WB", "outdoor — WB as-shot",
          "indoor — XYZ sans WB", "indoor — WB as-shot"], size=3.5)


In [ ]:
import cv2

# Charge tous les RAW en XYZ (réduits à MAX_SIDE pour tenir en mémoire)
def load_pool(files):
    pool = []
    for path, loc, cats in tqdm(files, desc="RAW -> XYZ"):
        xyz, illum = read_raw_to_xyz(path)
        if illum is None:
            illum = gray_world_white(xyz)
        h, w = xyz.shape[:2]
        scale = MAX_SIDE / max(h, w)
        if scale < 1:
            xyz = cv2.resize(xyz, (round(w * scale), round(h * scale)),
                             interpolation=cv2.INTER_AREA)
        pool.append({"xyz": xyz, "e": read_exposure(path), "illum": illum,
                     "loc": loc, "cats": cats, "name": path.stem})
    return pool


pool = load_pool(dng_files)
print(f"{len(pool)} images, expositions e de "
      f"{min(p['e'] for p in pool):.2e} à {max(p['e'] for p in pool):.2e}")


## 3. Simulation de reflets (Func. 1 du papier)

Pour une paire d'images XYZ `(i, j)` :

1. `j` est coupée en deux moitiés disjointes → source du **reflet** `r` et du
   **contexte** `c` (Sec. 3.3 : la photo contextuelle regarde la scène
   reflétée mais ne partage pas exactement son contenu — les moitiés
   disjointes modélisent ça gratuitement) ;
2. crop aléatoire de `i` → **transmission** `t` ;
3. **dé-exposition** par `e = s·g/n²` : les pixels redeviennent proportionnels
   à la luminance de la scène ;
4. simulation **géométrique** (Sec. 3.2 / B) : on échantillonne une **caméra**
   (FOV 40–90°, ~65° en moyenne comme le papier) et une **vitre inclinée**,
   puis on en dérive physiquement :
   - l'angle d'incidence *par pixel* → **réflectance de Fresnel exacte**
     (verre n = 1.52, deux interfaces) — plus forte côté rasant ;
   - une **homographie de rotation** `H = K·R·K⁻¹` : le reflet est vu sous un
     autre angle que la photo qui lui sert de source ;
   - un **défocus optique** : cercle de confusion calculé avec mise au point
     sur le sujet transmis, la scène reflétée étant virtuellement à
     `d_vitre + d_scène` — la plupart des reflets restent nets, comme le
     note le papier ;
   - un **fantôme** de double réflexion : les deux faces de la vitre
     réfléchissent presque autant (~4 % chacune), décalées de
     `2·h·tan(θ_t)·cos(θ_i)` (épaisseur du verre h, projeté en pixels) ;
   - la transmission est atténuée par `(1 − R)` (la vitre absorbe une part
     de la lumière transmise) ;
5. mélange **additif** `m = t + r` ;
6. **ré-exposition** : le pixel moyen vise `τ` (fonction de capture `C`) ;
7. **balance des blancs partagée** : le photographe règle pour son sujet →
   on balance sur l'**illuminant as-shot de la transmission** (CAT Bradford
   → D50). Le reflet garde donc sa dominante — bleu d'un extérieur au-dessus
   d'un intérieur tungstène, jaune d'un intérieur sur un extérieur — ce sont
   exactement les priors de couleur discutés dans le papier (Fig. 3a) ;
8. conversion **XYZ(D50) → sRGB linéaire**.

**Écart principal au papier** (`NORMALIZE_POWER = True`) : eux mélangent des
paires purement physiques et *cherchent* les mélanges utiles parmi 10⁸
candidats. Avec ~16 images, les ratios d'exposition FiveK (plein soleil vs
intérieur sombre) donnent presque toujours un reflet invisible ou écrasant.
On normalise donc chaque composante à luminance moyenne 1 et on échantillonne
un ratio de puissance reflet/transmission plausible (log-uniforme ~[0.3, 5],
encore atténué par Fresnel). Mets le flag à `False` pour la version purement
physique du papier (taux d'acceptation très faible).


In [ ]:
GLASS_IOR = 1.52      # indice de réfraction d'un verre courant
SENSOR_W = 0.036      # capteur plein format (m)


def _fresnel_single(cos_i, n=GLASS_IOR):
    """Fresnel non polarisé, une interface air->verre."""
    cos_i = np.clip(cos_i, 1e-6, 1.0)
    sin_t = np.sqrt(np.clip(1 - cos_i ** 2, 0, 1)) / n
    cos_t = np.sqrt(np.clip(1 - sin_t ** 2, 0, 1))
    rs = ((cos_i - n * cos_t) / (cos_i + n * cos_t)) ** 2
    rp = ((n * cos_i - cos_t) / (n * cos_i + cos_t)) ** 2
    return 0.5 * (rs + rp)


def fresnel_reflectance(cos_i, n=GLASS_IOR):
    """Vitre à deux interfaces (réflexions internes multiples) : 2R/(1+R)."""
    R = _fresnel_single(cos_i, n)
    return 2 * R / (1 + R)


def sample_scene(rng, size):
    """Tire une caméra + une vitre + des distances, et en dérive tous les
    paramètres géométriques du reflet (Sec. B, simplifié : une vitre plane)."""
    # caméra : FOV 40-90° (moyenne ~65° comme le papier, Sec. B.2)
    fov = float(rng.uniform(40, 90))
    f_px = (size / 2) / np.tan(np.radians(fov) / 2)

    # rayons par pixel + normale de la vitre inclinée de theta0
    ys, xs = np.mgrid[0:size, 0:size].astype(np.float32) - (size - 1) / 2
    d = np.stack([xs, ys, np.full_like(xs, f_px)], -1)
    d /= np.linalg.norm(d, axis=-1, keepdims=True)
    theta0 = np.radians(float(rng.uniform(0, 55)))
    phi = float(rng.uniform(0, 2 * np.pi))
    nrm = np.array([np.sin(theta0) * np.cos(phi),
                    np.sin(theta0) * np.sin(phi),
                    np.cos(theta0)], np.float32)
    cos_i = np.abs(d @ nrm)                      # angle d'incidence par pixel

    # défocus : mise au point sur le sujet transmis, la scène reflétée est
    # virtuellement à d_glass + distance derrière la vitre (miroir)
    f_m = f_px / size * SENSOR_W                 # focale en mètres
    N = float(rng.uniform(1.8, 8.0))             # ouverture
    d_focus = 10.0 ** float(rng.uniform(-0.3, 0.9))          # sujet : 0.5-8 m
    d_glass = float(rng.uniform(0.3, max(0.4, min(3.0, d_focus))))
    d_refl = d_glass + 10.0 ** float(rng.uniform(0.0, 1.7))  # 1-50 m derrière
    coc = f_m ** 2 / (N * max(d_focus - f_m, 1e-3)) * abs(d_refl - d_focus) / d_refl
    defocus_px = coc / SENSOR_W * size           # cercle de confusion en pixels

    # fantôme : décalage latéral de la réflexion sur la 2e face du verre
    h_glass = float(rng.uniform(0.003, 0.012))   # épaisseur du verre (m)
    sin_t = np.sin(theta0) / GLASS_IOR
    tan_t = sin_t / np.sqrt(1 - sin_t ** 2)
    ghost_px = f_px * 2 * h_glass * tan_t * np.cos(theta0) / d_glass

    return {"f_px": f_px, "theta0": theta0,
            "direction": (float(np.cos(phi)), float(np.sin(phi))),
            "R_map": fresnel_reflectance(cos_i)[..., None].astype(np.float32),
            "R1_center": float(_fresnel_single(np.cos(theta0))),
            "defocus_px": float(defocus_px), "ghost_px": float(ghost_px)}


def random_crop(img, size, rng):
    h, w = img.shape[:2]
    c = int(min(h, w) * rng.uniform(0.4, 1.0))
    y = int(rng.integers(0, h - c + 1))
    x = int(rng.integers(0, w - c + 1))
    crop = img[y:y + c, x:x + c]
    interp = cv2.INTER_AREA if c >= size else cv2.INTER_LINEAR
    return cv2.resize(crop, (size, size), interpolation=interp)


def split_reflection_context(img, rng):
    """Moitiés disjointes (Sec. 3.3) : l'une donne r, l'autre c."""
    h, w = img.shape[:2]
    if rng.random() < 0.5:
        halves = img[:, : w // 2], img[:, w // 2:]
    else:
        halves = img[: h // 2], img[h // 2:]
    i = int(rng.random() < 0.5)
    return halves[i], halves[1 - i]


def perspective_warp(img, scene, rng):
    """Le reflet est vu sous un autre angle que la photo source : homographie
    de rotation H = K R K^-1 autour de l'axe du plan d'incidence."""
    angle = float(rng.uniform(0.1, 0.5)) * scene["theta0"]
    if angle < 1e-3:
        return img
    size = img.shape[0]
    dx, dy = scene["direction"]
    R, _ = cv2.Rodrigues(np.array([-dy, dx, 0.0], np.float64) * angle)
    K = np.array([[scene["f_px"], 0, (size - 1) / 2],
                  [0, scene["f_px"], (size - 1) / 2],
                  [0, 0, 1]], np.float64)
    H = K @ R @ np.linalg.inv(K)
    return cv2.warpPerspective(img, H.astype(np.float32), (size, size),
                               borderMode=cv2.BORDER_REFLECT)


def disk_kernel(radius):
    y, x = np.mgrid[-radius:radius + 1, -radius:radius + 1]
    k = ((x ** 2 + y ** 2) <= radius ** 2).astype(np.float32)
    return k / k.sum()


def defocus(img, radius):
    """Flou de défocus : noyau disque (bokeh), pas gaussien."""
    if radius < 1:
        return img
    return cv2.filter2D(img, -1, disk_kernel(int(radius)))


def double_reflection(img, scene, rng):
    """Fantôme : les deux faces de la vitre réfléchissent presque autant
    (~4 % chacune) ; la 2e est décalée dans le plan d'incidence et pondérée
    par (1-R)^2 (deux traversées de la première interface)."""
    shift = scene["ghost_px"]
    if shift < 0.5:
        return img
    dx, dy = scene["direction"]
    M = np.float32([[1, 0, dx * shift], [0, 1, dy * shift]])
    ghost = cv2.warpAffine(img, M, img.shape[1::-1],
                           borderMode=cv2.BORDER_REFLECT)
    a = (1.0 - scene["R1_center"]) ** 2
    return (img + a * ghost) / (1.0 + a)


In [ ]:
def simulate_example(src_t, src_r, rng, patch=None, tau=None):
    """Func. 1 du papier. src_t, src_r = entrées du pool (xyz, e, illum...).
    Retourne (m, t, r, c) en sRGB linéaire."""
    patch = patch or PATCH
    tau = tau or TAU

    # 1-2) découpes : t depuis l'image i ; (r, c) = moitiés disjointes de j
    r_src, c_src = split_reflection_context(src_r["xyz"], rng)
    t = random_crop(src_t["xyz"], patch, rng)
    r = random_crop(r_src, patch, rng)
    c = random_crop(c_src, patch, rng)

    # 3) dé-exposition : pixels proportionnels à la luminance de scène
    t, r, c = t / src_t["e"], r / src_r["e"], c / src_r["e"]

    if NORMALIZE_POWER:
        # écart au papier (voir markdown) : ratio de puissance échantillonné
        rho = 10.0 ** float(rng.uniform(-0.5, 0.7))
        t = t / (t[..., 1].mean() + 1e-8)
        scale = rho / (r[..., 1].mean() + 1e-8)
        r, c = r * scale, c * scale

    # 4) simulation géométrique : caméra + vitre échantillonnées (Sec. B)
    scene = sample_scene(rng, patch)
    if rng.random() < 0.5:
        r = r[:, ::-1].copy()                    # miroir (c reste une vue directe)
    r = perspective_warp(r, scene, rng)          # vu sous un autre angle
    r = defocus(r, int(round(min(scene["defocus_px"], 12))))
    r = double_reflection(r, scene, rng)         # fantôme des deux faces
    r = r * scene["R_map"]                       # Fresnel par pixel
    t = t * (1.0 - scene["R_map"])               # la vitre atténue aussi t

    # 5) mélange additif : la lumière s'additionne sur le capteur
    m = t + r

    # 6) ré-exposition (Func. S1 simplifié : pas de règle de saturation,
    #    le filtrage rejettera les mélanges trop saturés)
    e = tau / (m[..., 1].mean() + 1e-8)
    m, t, r, c = m * e, t * e, r * e, c * e
    m = np.clip(m, 0, 1)                         # saturation du capteur

    # 7) WB partagée : illuminant as-shot de la transmission (le photographe
    #    règle pour son sujet ; le reflet garde sa dominante, Fig. 3a)
    # 8) XYZ(D50) -> sRGB linéaire
    M = XYZ_D50_TO_SRGB @ cat_matrix(src_t["illum"])
    m, t, r, c = (np.clip(apply_matrix(x, M), 0, 1).astype(np.float32)
                  for x in (m, t, r, c))
    return {"m": m, "t": t, "r": r, "c": c}


In [ ]:
def is_useful(ex):
    """Recherche du papier (Sec. D) réduite à quelques heuristiques :
    bien exposé, reflet visible mais pas destructeur, structures présentes."""
    Ym = luminance(ex["m"]).mean()
    if not 0.04 < Ym < 0.6:                       # exposition correcte
        return False
    vis = luminance(ex["r"]).mean() / (Ym + 1e-8)
    if not 0.04 < vis < 0.6:                      # visibilité du reflet
        return False
    if (ex["m"].max(axis=-1) > 0.99).mean() > 0.08:   # saturation
        return False
    if luminance(ex["t"]).std() < 0.015:          # transmission plate/noyée
        return False
    if luminance(ex["r"]).std() < 0.008:          # reflet sans structure
        return False
    return True


## 3 bis. Choisir des paires réalistes (Sec. D du papier)

Le papier n'apparie pas les images au hasard : le verre est placé avec
intention dans le monde (vitrines, musées, voitures...), ce qui crée des
priors. Ils les obtiennent en cherchant parmi 10⁸ mélanges physiques ; ici on
les encode dans un **score de paire** à partir des catégories FiveK et des
illuminants :

- une vitre sépare presque toujours un **intérieur d'un extérieur** ;
- un **reflet extérieur** est d'autant plus plausible/fort qu'il fait jour
  (lumière puissante → reflets colorés de scènes entières, souvent bleutés) ;
- un **reflet intérieur** est crédible surtout en lumière artificielle,
  au-dessus d'une scène extérieure sombre (crépuscule, nuit, vitrine le
  soir) — la lumière intérieure est faible ;
- des **illuminants différents** donnent les mélanges de couleurs
  intéressants qui justifient tout le pipeline RAW (Fig. 3a).

`sample_realistic_pair` tire ensuite les paires proportionnellement à leur
score. (Pour un très gros pool, remplacer l'énumération O(n²) par du tirage
de candidats + rejet.)


In [ ]:
def pair_score(src_t, src_r):
    """Score de réalisme d'une paire (transmission t, reflet r)."""
    ct, cr = src_t["cats"], src_r["cats"]
    loc_t, loc_r = ct.get("location"), cr.get("location")
    s = 1.0

    # 1) une vitre sépare presque toujours deux espaces différents
    if loc_t != loc_r and "unknown" not in (loc_t, loc_r):
        s *= 4.0
    elif loc_t == loc_r and loc_t != "unknown":
        s *= 0.3

    # 2) reflet extérieur : d'autant plus fort qu'il fait jour
    if loc_r == "outdoor":
        s *= {"day": 2.0, "dusk": 1.5, "night": 0.7}.get(cr.get("time"), 1.0)

    # 3) reflet intérieur : lumière artificielle, sur une scène sombre
    if loc_r == "indoor":
        if cr.get("light") == "artificial":
            s *= 1.5
        s *= {"night": 2.0, "dusk": 2.0, "day": 0.7}.get(ct.get("time"), 1.0)

    # 4) illuminants différents -> mélange de couleurs réaliste et informatif
    it = src_t["illum"] / src_t["illum"].sum()
    ir = src_r["illum"] / src_r["illum"].sum()
    s *= 1.0 + 4.0 * min(float(np.abs(it - ir).sum()), 0.25)
    return s


ALL_PAIRS = [(i, j) for i in range(len(pool))
             for j in range(len(pool)) if i != j]
PAIR_WEIGHTS = [pair_score(pool[i], pool[j]) for i, j in ALL_PAIRS]


def sample_realistic_pair(rnd):
    """(index transmission, index reflet), ~ proportionnel au score."""
    return rnd.choices(ALL_PAIRS, weights=PAIR_WEIGHTS, k=1)[0]


print("paires les mieux notées :")
for w, (i, j) in sorted(zip(PAIR_WEIGHTS, ALL_PAIRS), reverse=True)[:5]:
    print(f"  score {w:5.2f} : t = {pool[i]['name']} ({pool[i]['loc']})  "
          f"<-  reflet = {pool[j]['name']} ({pool[j]['loc']})")


In [ ]:
# Démo : on tire des paires réalistes jusqu'à trouver un mélange accepté
ex = None
for _ in range(300):
    ia, ib = sample_realistic_pair(rnd)
    cand = simulate_example(pool[ia], pool[ib], rng)
    if is_useful(cand):
        ex = cand
        break
assert ex is not None, "aucun mélange accepté — relance la cellule"
print(f"t = {pool[ia]['name']} ({pool[ia]['loc']}), "
      f"r = {pool[ib]['name']} ({pool[ib]['loc']})")
show_row([ex["m"], ex["t"], ex["r"], ex["c"]],
         ["mixture m = t + r", "transmission t", "reflet r", "contexte c"])


## 4. Pipeline ISP : du linéaire au sRGB affichable

C'est elle qui va « finir » le dataset en JPEG. Pour un DNG complet (ou une
sortie de modèle en linéaire), il reste les étapes ACR 5–8 : balance des
blancs, conversion RGB, tone mapping, gamma. Version simplifiée :

1. **WB** : illuminant as-shot du DNG (fallback gray-world) + CAT Bradford
   → D50 — c'est le chemin « color matrices » de la spec DNG que le papier
   utilise (XYZ d'abord, WB ensuite) ;
2. **XYZ(D50) → sRGB linéaire** ;
3. **auto-exposition** : moyenne géométrique de la luminance → gris moyen
   (comme un posemètre) ;
4. **courbe de tons** : Reinhard étendu — remonte les ombres, roll-off doux
   des hautes lumières *(dans ACR : courbe propriétaire spatialement variable,
   c'est justement pourquoi on ne peut pas simuler des reflets sur des JPEG)* ;
5. **gamma sRGB** → 8 bits.


In [ ]:
def tone_curve(x, white=4.0):
    """Reinhard étendu : x*(1 + x/white^2)/(1 + x)."""
    return x * (1.0 + x / white ** 2) / (1.0 + x)


def isp(lin_srgb, ev=0.0, tone=True):
    """sRGB linéaire -> image 8 bits affichable."""
    x = np.clip(lin_srgb, 0, None) * 2.0 ** ev
    if tone:
        x = tone_curve(x)
    return (srgb_encode(np.clip(x, 0, 1)) * 255 + 0.5).astype(np.uint8)


def dng_to_srgb(path, ev=0.0, tone=True):
    """Pipeline complète DNG -> sRGB 8 bits."""
    xyz, illum = read_raw_to_xyz(path)
    if illum is None:
        illum = gray_world_white(xyz)
    lin = apply_matrix(xyz, XYZ_D50_TO_SRGB @ cat_matrix(illum))
    lin = auto_expose(np.clip(lin, 0, None))
    return isp(lin, ev=ev, tone=tone)


# Comparaison : notre ISP vs le rendu par défaut de LibRaw (WB appareil)
p = dng_files[0][0]
ours = dng_to_srgb(p)
with rawpy.imread(str(p)) as raw:
    ref = raw.postprocess(use_camera_wb=True, half_size=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, img, title in zip(axes, [ours, ref], ["notre ISP", "LibRaw (défaut)"]):
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Génération du dataset (JPEG sRGB)

Boucle : tirer une paire réaliste (`sample_realistic_pair`), simuler **en
linéaire**, filtrer, puis finir chaque composante avec la **même ISP**
(tone mapping + gamma, sans ré-exposition par composante pour rester
cohérent) et sauver `ex_XXXX_{m,t,r,c}.jpg`.

**Important** : après tone mapping, l'additivité `m = t + r` n'est **plus**
vérifiée — et c'est normal. Elle n'était nécessaire qu'au moment de la
synthèse, en linéaire. Le résultat : un dataset sRGB « comme des vraies
photos JPEG », mais dont le mélange sous-jacent était photométriquement
correct. (`SAVE_LINEAR = True` pour garder aussi les `.npz` linéaires,
par exemple pour entraîner plus tard un modèle RAW comme celui du papier.)


In [ ]:
def save_jpeg(path, lin_srgb):
    img = isp(lin_srgb)                    # tone mapping + gamma sRGB partagés
    cv2.imwrite(str(path), img[..., ::-1],
                [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])


n_saved = n_tried = 0
pbar = tqdm(total=N_EXAMPLES, desc="simulation")
while n_saved < N_EXAMPLES and n_tried < 200 * N_EXAMPLES:
    n_tried += 1
    ia, ib = sample_realistic_pair(rnd)
    ex = simulate_example(pool[ia], pool[ib], rng)
    if not is_useful(ex):
        continue
    stem = f"ex_{n_saved:04d}"
    for k in "mtrc":
        save_jpeg(OUT_DIR / f"{stem}_{k}.jpg", ex[k])
    if SAVE_LINEAR:
        np.savez_compressed(OUT_DIR / f"{stem}.npz",
                            **{k: v.astype(np.float16) for k, v in ex.items()},
                            t_name=pool[ia]["name"], r_name=pool[ib]["name"])
    n_saved += 1
    pbar.update(1)
pbar.close()
print(f"{n_saved} exemples acceptés sur {n_tried} essais "
      f"({n_saved / max(n_tried, 1):.0%} d'acceptation) -> {OUT_DIR}")


In [ ]:
def show_examples(n=3, root=None, start=0):
    """Affiche n exemples du dataset JPEG : une ligne (m | t | r | c) chacun.

    n     : nombre d'exemples à afficher
    root  : dossier des JPEG (OUT_DIR par défaut)
    start : index du premier exemple (pour paginer : show_examples(3, start=3))
    """
    root = Path(root) if root else OUT_DIR
    stems = sorted({p.name[:-6] for p in root.glob("ex_*_m.jpg")})[start:start + n]
    if not stems:
        print(f"aucun exemple trouvé dans {root}")
        return
    titles = ["mixture m", "transmission t", "reflet r", "contexte c"]
    fig, axes = plt.subplots(len(stems), 4, figsize=(13, 3.4 * len(stems)),
                             squeeze=False)
    for row, stem in zip(axes, stems):
        for ax, k, title in zip(row, "mtrc", titles):
            img = cv2.imread(str(root / f"{stem}_{k}.jpg"))
            ax.imshow(img[..., ::-1])
            ax.set_title(f"{stem} — {title}", fontsize=9)
            ax.axis("off")
    plt.tight_layout()
    plt.show()


show_examples(3)


## 6. Dataset PyTorch pour l'entraînement

Le modèle de base du papier prend `(m, c)` en entrée et prédit `(t, r)`
indépendamment (sans imposer `t + r = m`, pour découpler les échecs).
Ci-dessous : un `Dataset` qui charge les JPEG en `[0,1]`, avec crop et flip
aléatoires. La dernière vérification illustre que l'additivité ne tient plus
en sRGB tone-mappé — attendu, la physique a déjà joué son rôle à la synthèse.
Note : depuis l'ajout de l'atténuation `(1-R)`, `t` est la transmission *à
travers la vitre* — c'est bien elle qu'on veut récupérer.


In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset


class FiveKReflections(Dataset):
    """(mixture, context) -> (transmission, reflection), en sRGB [0,1]."""

    def __init__(self, root, patch=224, train=True):
        self.root = Path(root)
        self.stems = sorted(p.name[:-6] for p in self.root.glob("ex_*_m.jpg"))
        self.patch = patch
        self.train = train

    def _load(self, stem, k):
        img = cv2.imread(str(self.root / f"{stem}_{k}.jpg"))
        return img[..., ::-1].astype(np.float32) / 255.0

    def __len__(self):
        return len(self.stems)

    def __getitem__(self, idx):
        stem = self.stems[idx]
        m, t, r, c = (self._load(stem, k) for k in "mtrc")
        if self.train:
            p = self.patch
            y = np.random.randint(0, m.shape[0] - p + 1)
            x = np.random.randint(0, m.shape[1] - p + 1)
            m, t, r = (a[y:y + p, x:x + p] for a in (m, t, r))
            c = cv2.resize(c, (p, p), interpolation=cv2.INTER_AREA)
            if np.random.rand() < 0.5:
                m, t, r, c = (a[:, ::-1] for a in (m, t, r, c))
        to_tensor = lambda a: torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1)
        return {"mixture": to_tensor(m), "context": to_tensor(c),
                "transmission": to_tensor(t), "reflection": to_tensor(r)}


ds = FiveKReflections(OUT_DIR)
batch = next(iter(DataLoader(ds, batch_size=min(4, len(ds)), shuffle=True)))
print({k: tuple(v.shape) for k, v in batch.items()})

# En sRGB tone-mappé l'additivité n'est PLUS vérifiée — attendu : le tone
# mapping est non linéaire. Elle n'était requise qu'à la synthèse, en linéaire.
m, t, r = (ds._load(ds.stems[0], k) for k in "mtr")
diff = np.abs(np.clip(t + r, 0, 1) - m)
print(f"|m - clip(t+r)| en sRGB : moyenne = {diff.mean():.3f} (non nul, normal)")


## 7. Ce qui manque par rapport au papier — et pistes pour la suite

**Simplifications restantes** :
- **Échelle** : eux utilisent ~25 000 sources (MIT5K + RAISE + panoramas Laval
  Indoor) et cherchent parmi 10⁸ candidats ; nous, quelques DNG FiveK, un
  score de paires et un ratio de puissance échantillonné (`NORMALIZE_POWER`).
- **Géométrie** : une seule vitre plane et une homographie de rotation pour la
  perspective — pas la vraie projection double (caméra virtuelle derrière le
  miroir) ni les poses contraintes de la Sec. B ; Fresnel par rayon mais
  vitre non polarisante idéale (pas de teinte, saleté, rayures).
- **Couleur** : illuminant as-shot du DNG au lieu de la WB ACR complète
  (Func. S2/S9), matrice `ColorMatrix` seule au lieu de l'interpolation
  entre les deux illuminants de calibration de la spec DNG.
- **Ré-exposition** : pas de règle de saturation (Func. S1) — on filtre au
  lieu de préserver les pixels saturés.
- Pas d'écriture DNG des sorties, pas de modèle 2048p ni d'upsampler.

**Pistes pour ton projet de reflection removal** :
1. **Monte en volume** : augmente `N_OUTDOOR/N_INDOOR` (les 5 000 DNG ≈ 50 Go),
   génère 10-100k exemples ; le papier montre que la qualité des données RAW
   compte plus que l'architecture (+40 pts SSIM vs ~20 pts entre modèles).
2. **Entraîne un premier modèle** : U-Net léger qui prend `m` (et `c` en
   modulation par canal, à la StyleGAN) et prédit `(t, r)` — pertes L1 +
   perceptuelle (VGG) directement sur les JPEG en `[0,1]`. Le message du
   papier reste valable en sRGB : ce qui compte, c'est que le *mélange* ait
   été simulé physiquement en RAW, pas le format d'entraînement. Si un jour
   tu veux un modèle RAW de bout en bout comme le leur, régénère avec
   `SAVE_LINEAR = True` et entraîne sur les `.npz` linéaires.
3. **Priors de saturation** : implémente la règle Func. S1 (garder saturé ce
   qui est saturé) — le papier note que le vrai problème devient alors du
   *hole filling*.
4. **Expert TIFF16** : FiveK fournit 5 retouches expertes par image
   (`urls["tiff16"]["c"]`) — utile si tu veux aussi apprendre une ISP neurale
   (RAW → rendu expert) au lieu de notre ISP à la main.
5. **Upsampler** : le papier traite à 256p puis upsample par masquage de
   features sur pyramide gaussienne (Sec. 4.2) — indispensable si tu veux des
   sorties pleine résolution.
